# Soil Image Model Comparison: CNN vs ResNet

This notebook is designed for Google Colab. It trains and compares:
- a custom CNN
- a ResNet18 transfer learning model

Both models use the same clean split policy:
- source: `Orignal-Dataset` only
- train/val/test: stratified `70/15/15`
- augmentation: train only
- effective train size: `8000` samples per epoch via online augmentation


## 1. Runtime Setup

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.datasets.folder import default_loader

SEED = 42
TARGET_TRAIN_SAMPLES = 8000
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Point This Notebook at the Project Folder

If you upload the project to Colab, change `PROJECT_ROOT` below as needed.

In [ ]:
PROJECT_ROOT = Path(r'C:\Users\manis\OneDrive\Desktop\PA-project')
DATA_ROOT = PROJECT_ROOT / 'Dataset' / 'soil_image_datset' / 'Soil-image-dataset' / 'Orignal-Dataset'
MODELS_DIR = PROJECT_ROOT / 'Models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_ROOT.exists(), f'Missing dataset directory: {DATA_ROOT}'
print(DATA_ROOT)


## 3. Dataset Helpers

In [ ]:
class SoilImageDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.targets = [label for _, label in samples]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = default_loader(image_path)
        return self.transform(image), label


class AugmentedOversampledDataset(Dataset):
    def __init__(self, samples, transform, target_size):
        self.samples = samples
        self.targets = [label for _, label in samples]
        self.transform = transform
        self.target_size = target_size

    def __len__(self):
        return self.target_size

    def __getitem__(self, index):
        image_path, label = self.samples[index % len(self.samples)]
        image = default_loader(image_path)
        return self.transform(image), label


def collect_class_names(root_dir):
    return sorted(path.name for path in root_dir.iterdir() if path.is_dir())


def collect_samples(root_dir, class_names):
    samples = []
    for class_index, class_name in enumerate(class_names):
        class_dir = root_dir / class_name
        for image_path in sorted(path for path in class_dir.iterdir() if path.is_file()):
            samples.append((str(image_path), class_index))
    return samples


def summarize_split(split_name, samples, class_names):
    counts = Counter(label for _, label in samples)
    print(f'\n{split_name}: {len(samples)} images')
    for class_index, class_name in enumerate(class_names):
        print(f'  {class_name}: {counts.get(class_index, 0)}')


def build_datasets(data_root):
    class_names = collect_class_names(data_root)
    all_samples = collect_samples(data_root, class_names)
    labels = [label for _, label in all_samples]

    train_samples, temp_samples = train_test_split(
        all_samples,
        test_size=0.30,
        random_state=SEED,
        stratify=labels,
    )
    temp_labels = [label for _, label in temp_samples]
    val_samples, test_samples = train_test_split(
        temp_samples,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_labels,
    )

    summarize_split('Train source', train_samples, class_names)
    summarize_split('Validation', val_samples, class_names)
    summarize_split('Test', test_samples, class_names)
    print(f'\nTrain augmented target: {TARGET_TRAIN_SAMPLES} samples per epoch')

    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    return (
        AugmentedOversampledDataset(train_samples, train_transform, TARGET_TRAIN_SAMPLES),
        SoilImageDataset(val_samples, eval_transform),
        SoilImageDataset(test_samples, eval_transform),
        class_names,
        len(train_samples),
    )


train_dataset, val_dataset, test_dataset, class_names, train_source_count = build_datasets(DATA_ROOT)
label_encoder = LabelEncoder()
label_encoder.fit(class_names)
joblib.dump(label_encoder, MODELS_DIR / 'soil_label_encoder_7class.pkl')


## 4. DataLoaders

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 5. Model Definitions

In [ ]:
class SoilCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_resnet(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


## 6. Training and Evaluation Helpers

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    predictions = []
    targets = []
    probs = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            predictions.extend(outputs.argmax(dim=1).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
    avg_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(targets, predictions)
    return avg_loss, acc, np.array(targets), np.array(predictions), np.array(probs)


def train_model(model_name, model, epochs, learning_rate):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_state = None
    best_val_acc = 0.0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            correct += outputs.argmax(dim=1).eq(labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"{model_name} | Epoch {epoch + 1}/{epochs} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_loss, test_acc, y_true, y_pred, y_prob = evaluate(model, test_loader, criterion)
    return {
        'model_name': model_name,
        'model': model,
        'history': pd.DataFrame(history),
        'best_epoch': best_epoch,
        'best_val_acc': best_val_acc,
        'test_loss': test_loss,
        'test_acc': test_acc,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }


## 7. Train CNN

In [ ]:
cnn_result = train_model('CNN', SoilCNN(len(class_names)), epochs=20, learning_rate=1e-3)
torch.save(cnn_result['model'].state_dict(), MODELS_DIR / 'soil_cnn_updated_model.pth')


## 8. Train ResNet18

In [ ]:
resnet_result = train_model('ResNet18', build_resnet(len(class_names)), epochs=15, learning_rate=1e-4)
torch.save(resnet_result['model'].state_dict(), MODELS_DIR / 'soil_resnet_updated_model.pth')


## 9. Compare Metrics

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': cnn_result['model_name'],
        'Best Validation Accuracy': cnn_result['best_val_acc'],
        'Test Accuracy': cnn_result['test_acc'],
        'Best Epoch': cnn_result['best_epoch'],
    },
    {
        'Model': resnet_result['model_name'],
        'Best Validation Accuracy': resnet_result['best_val_acc'],
        'Test Accuracy': resnet_result['test_acc'],
        'Best Epoch': resnet_result['best_epoch'],
    },
])
comparison_df


In [ ]:
plt.figure(figsize=(8, 5))
plot_df = comparison_df.melt(id_vars='Model', value_vars=['Best Validation Accuracy', 'Test Accuracy'])
sns.barplot(data=plot_df, x='Model', y='value', hue='variable')
plt.ylim(0, 1)
plt.title('CNN vs ResNet18 Accuracy Comparison')
plt.ylabel('Accuracy')
plt.show()


## 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for result in [cnn_result, resnet_result]:
    history = result['history']
    axes[0].plot(history['epoch'], history['train_acc'], label=f"{result['model_name']} Train")
    axes[0].plot(history['epoch'], history['val_acc'], label=f"{result['model_name']} Val")
    axes[1].plot(history['epoch'], history['train_loss'], label=f"{result['model_name']} Train")
    axes[1].plot(history['epoch'], history['val_loss'], label=f"{result['model_name']} Val")

axes[0].set_title('Accuracy Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].set_title('Loss Curves')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


## 11. Confusion Matrices

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

plot_confusion(cnn_result['y_true'], cnn_result['y_pred'], 'CNN Test Confusion Matrix')
plot_confusion(resnet_result['y_true'], resnet_result['y_pred'], 'ResNet18 Test Confusion Matrix')


## 12. Save Comparison Outputs

In [ ]:
cnn_metrics = {
    'architecture': 'CNN',
    'class_names': class_names,
    'counts': {'train': TARGET_TRAIN_SAMPLES, 'train_source_images': train_source_count, 'val': len(val_dataset), 'test': len(test_dataset)},
    'best_epoch': cnn_result['best_epoch'],
    'best_val_accuracy': cnn_result['best_val_acc'],
    'test_accuracy': cnn_result['test_acc'],
    'model_path': str(MODELS_DIR / 'soil_cnn_updated_model.pth'),
}

resnet_metrics = {
    'architecture': 'ResNet18',
    'class_names': class_names,
    'counts': {'train': TARGET_TRAIN_SAMPLES, 'train_source_images': train_source_count, 'val': len(val_dataset), 'test': len(test_dataset)},
    'best_epoch': resnet_result['best_epoch'],
    'best_val_accuracy': resnet_result['best_val_acc'],
    'test_accuracy': resnet_result['test_acc'],
    'model_path': str(MODELS_DIR / 'soil_resnet_updated_model.pth'),
}

with open(MODELS_DIR / 'soil_cnn_updated_model_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(cnn_metrics, f, indent=2)

with open(MODELS_DIR / 'soil_resnet_updated_model_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(resnet_metrics, f, indent=2)

comparison_df.to_csv(MODELS_DIR / 'soil_image_model_comparison.csv', index=False)
comparison_df
